In [1]:
!git clone https://github.com/dave123981/commerce-recommendation-engine.git
%cd commerce-recommendation-engine
!pip install -r requirements.txt -q


Cloning into 'commerce-recommendation-engine'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 186 (delta 103), reused 91 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 62.33 KiB | 2.60 MiB/s, done.
Resolving deltas: 100% (103/103), done.
/content/commerce-recommendation-engine


In [2]:
!pwd

/content/commerce-recommendation-engine


In [ ]:
!pip install -r requirements.txt

In [4]:
!pip install -q kagglehub
import kagglehub

!python data/download_data.py
!python data/build_interactions.py

100% 197M/197M [00:01<00:00, 172MB/s]
Extracting files...
kagglehub cached the dataset at /root/.cache/kagglehub/datasets/psparks/instacart-market-basket-analysis/versions/1
Copied CSVs to /content/commerce-recommendation-engine/data/raw
Built 31,803,792 interactions | 175,071 users | 20,094 items | 3,223,536 orders
Saved to /content/commerce-recommendation-engine/data/interactions.csv


In [5]:
import pandas as pd

interactions = pd.read_csv("data/interactions.csv", parse_dates=["timestamp"])

print(interactions.shape)
print(interactions.nunique())
print("sparsity:", 1 - len(interactions) / (interactions.user_id.nunique() * interactions.item_id.nunique()))
print("orders per user:", interactions.groupby("user_id").order_id.nunique().describe())
print("purchases per item:", interactions.groupby("item_id").order_id.nunique().describe())

(31803792, 5)
user_id       175071
item_id        20094
order_id     3223536
timestamp        366
quantity           1
dtype: int64
sparsity: 0.9909593783559679
orders per user: count    175071.000000
mean         18.412735
std          17.131480
min           1.000000
25%           7.000000
50%          12.000000
75%          23.000000
max         100.000000
Name: order_id, dtype: float64
purchases per item: count     20094.000000
mean       1582.750672
std        7520.228193
min         100.000000
25%         183.000000
50%         371.000000
75%        1014.000000
max      476386.000000
Name: order_id, dtype: float64


In [6]:
import data.build_interactions as bi
interactions = bi.build_interactions(min_orders_per_user=10, min_purchases_per_item=50)
interactions.to_csv("data/interactions.csv", index=False)

In [7]:

from src.metrics import time_based_split

train, test = time_based_split(interactions, test_frac=0.2)
train.to_csv("data/train.csv", index=False)
test.to_csv("data/test.csv", index=False)
print(len(train), len(test))

22297249 5520463


In [9]:
from src.v1_popularity import PopularityRecommender
from src.metrics import evaluate_model

model = PopularityRecommender(half_life_days=30).fit(train)
metrics = evaluate_model(model, test, k=10)
print(metrics)

{'precision@10': 0.042816720496663165, 'recall@10': 0.014891039007152738, 'map@10': 0.017301025888708167, 'ndcg@10': 0.04700983159132022}


In [ ]:
from src.v2_association import AssociationRecommender
from src.metrics import evaluate_model

model_v2 = AssociationRecommender(min_support=0.001, min_confidence=0.2, min_lift=1.1).fit(train)
n_items_with_rules = len(model_v2._rules_map)
print(f"items with rules: {n_items_with_rules} / {train.item_id.nunique()}")
metrics_v2 = evaluate_model(model_v2, test, k=10)
print(metrics_v2)

items with rules: 16 / 24865


In [9]:
from src.v3_collaborative import CollaborativeRecommender
from src.metrics import evaluate_model

model_v3 = CollaborativeRecommender(n_neighbors=20).fit(train)
metrics_v3 = evaluate_model(model_v3, test, k=10)
print(metrics_v3)



{'precision@10': 0.05130262386328732, 'recall@10': 0.01855248251402882, 'map@10': 0.021867570341040896, 'ndcg@10': 0.05720287551934992}


In [ ]:
from src.v4_matrix_factorization import MatrixFactorizationRecommender

model_v4 = MatrixFactorizationRecommender(n_factors=50).fit(train)
metrics_v4 = evaluate_model(model_v4, test, k=10)
print(metrics_v4)